# NB-03: 3D Rotary Position Embeddings

## Learning Objectives
- Understand how 1D RoPE base frequencies are computed using `precompute_freqs_cis` and why float64 precision is required
- Trace the three-axis band split for 3D video (f, h, w) and understand why the temporal axis absorbs the rounding remainder
- See how the 3D frequency grid is assembled from three 1D tables and applied to Q/K via `rope_apply`
- Compare base DiT grid-based RoPE with S2V per-token RoPE (`rope_precompute`)

## Prerequisites
- **Prior notebooks:** NB-01 (RMSNorm -- used in Q/K normalization before RoPE), NB-02 (QKV projections -- RoPE applies to Q and K)
- **Assumed concepts:** Complex numbers, Euler's formula, rotation matrices

## Concept Map
- `precompute_freqs_cis` -> used per-axis in `precompute_freqs_cis_3d`
- `rope_apply` -> used in `SelfAttention.forward` (NB-04)
- 3D grid assembly -> used in `WanModel.forward` (NB-08, Track 2)
- S2V `rope_precompute` -> covered in depth in NB-14, Track 5

In [1]:
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios (up to 6 levels up).
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for DiT primitive demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
_diffsynth_stub = types.ModuleType("diffsynth")
_models_stub = types.ModuleType("diffsynth.models")
sys.modules["diffsynth"] = _diffsynth_stub
sys.modules["diffsynth.models"] = _models_stub
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_dit.py directly, bypassing the broken diffsynth/__init__.py chain
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit
_spec.loader.exec_module(dit)

# Import symbols used in this notebook
from diffsynth.models.wan_video_dit import precompute_freqs_cis, precompute_freqs_cis_3d, rope_apply
import torch
import torch.nn as nn
from einops import rearrange

print("Setup complete.")

Project root: /home/basheer/Signapse/Codes/wan22-architecture-course
Setup complete.


## 1. 1D RoPE: The Base Function

**WHY:** RoPE (Rotary Position Embeddings) encodes position by rotating pairs of dimensions by position-dependent angles. Nearby positions have similar rotations, enabling the model to learn relative position awareness through the dot-product of rotated query and key vectors.

**HOW:** `precompute_freqs_cis` computes a lookup table of complex rotation angles:
1. Compute frequency scales via a geometric series: `freqs[k] = 1 / (theta^(2k/dim))` -- from high frequency (fast oscillation) down to low frequency (slow oscillation)
2. Form the outer product: `freqs[position, k] = position * freqs[k]` -- the rotation angle for position `p` at frequency band `k`
3. Call `torch.polar(ones, freqs)` -- converts each angle to a unit complex number `e^(i * angle)`

The result is a `[positions, dim//2]` tensor of complex128 values. Each position maps to `dim//2` rotation angles.

Source: `diffsynth/models/wan_video_dit.py`, line 83

In [2]:
# Source: diffsynth/models/wan_video_dit.py, line 83
head_dim = 96  # reduced config: dim=384, num_heads=4

# 1D RoPE for a single axis
freqs_1d = precompute_freqs_cis(head_dim, end=1024)
assert freqs_1d.shape == torch.Size([1024, head_dim // 2])
assert freqs_1d.dtype == torch.complex128
print(f"1D freqs shape: {freqs_1d.shape}  (positions x dim_pairs)")
print(f"dtype: {freqs_1d.dtype}  (complex128 -- from torch.polar with float64)")
print(f"First position (p=0) magnitudes: {freqs_1d[0].abs()[:3]} (all 1.0 -- unit circle)")
print(f"Second position (p=1) real parts: {freqs_1d[1].real[:3]}")

1D freqs shape: torch.Size([1024, 48])  (positions x dim_pairs)
dtype: torch.complex128  (complex128 -- from torch.polar with float64)
First position (p=0) magnitudes: tensor([1., 1., 1.], dtype=torch.float64) (all 1.0 -- unit circle)
Second position (p=1) real parts: tensor([0.5403, 0.6783, 0.7768], dtype=torch.float64)


## 2. Three-Axis Band Split

Video tokens have three positional axes: **temporal** (frame index F), **height** (spatial row H), and **width** (spatial column W). `precompute_freqs_cis_3d` splits `head_dim` into three bands and creates independent 1D frequency tables for each:

- `f_dim = head_dim - 2 * (head_dim // 3)` -- temporal (absorbs integer division remainder)
- `h_dim = head_dim // 3` -- spatial height
- `w_dim = head_dim // 3` -- spatial width

**CRITICAL (Pitfall 2 -- Reduced Config vs Production):**

With the reduced config `head_dim=96`, the split is **perfectly equal** `(32, 32, 32)` because 96 divides evenly by 3. In production (`head_dim=128`), the split is **unequal** `(44, 42, 42)` -- the temporal axis absorbs the remainder (`128 - 2*42 = 44`). This gives the temporal axis slightly more representational capacity, which makes sense because video has more temporal variation than spatial.

Source: `diffsynth/models/wan_video_dit.py`, line 75

In [3]:
# Source: diffsynth/models/wan_video_dit.py, line 75
# Reduced config: head_dim=96 -- EQUAL bands (pedagogical difference from production)
head_dim = 96
f_dim = head_dim - 2 * (head_dim // 3)  # 96 - 2*32 = 32
h_dim = head_dim // 3                    # 32
w_dim = head_dim // 3                    # 32
assert f_dim + h_dim + w_dim == head_dim
print(f"Reduced config (head_dim={head_dim}): f={f_dim}, h={h_dim}, w={w_dim}  (equal)")

# Production config: head_dim=128 -- UNEQUAL bands
head_dim_prod = 128
f_dim_prod = head_dim_prod - 2 * (head_dim_prod // 3)  # 128 - 2*42 = 44
h_dim_prod = head_dim_prod // 3                          # 42
w_dim_prod = head_dim_prod // 3                          # 42
assert f_dim_prod + h_dim_prod + w_dim_prod == head_dim_prod
print(f"Production (head_dim={head_dim_prod}):  f={f_dim_prod}, h={h_dim_prod}, w={w_dim_prod}  (temporal absorbs remainder)")

Reduced config (head_dim=96): f=32, h=32, w=32  (equal)
Production (head_dim=128):  f=44, h=42, w=42  (temporal absorbs remainder)


## 3. Calling precompute_freqs_cis_3d

The function calls `precompute_freqs_cis` three times with the appropriate dimensions for each axis. Each returned tensor is a separate 1D frequency table with `end=1024` rows, covering up to 1024 positions per axis.

In [4]:
# Source: diffsynth/models/wan_video_dit.py, line 75
head_dim = 96  # reduced config
f_freqs, h_freqs, w_freqs = precompute_freqs_cis_3d(head_dim)
assert f_freqs.shape == torch.Size([1024, 16])  # f_dim//2 = 32//2 = 16
assert h_freqs.shape == torch.Size([1024, 16])  # h_dim//2 = 32//2 = 16
assert w_freqs.shape == torch.Size([1024, 16])  # w_dim//2 = 32//2 = 16
print(f"f_freqs: {f_freqs.shape}  (temporal)")
print(f"h_freqs: {h_freqs.shape}  (height)")
print(f"w_freqs: {w_freqs.shape}  (width)")
print(f"Total dim pairs: {f_freqs.shape[1] + h_freqs.shape[1] + w_freqs.shape[1]}  (== head_dim//2 = {head_dim//2})")

f_freqs: torch.Size([1024, 16])  (temporal)
h_freqs: torch.Size([1024, 16])  (height)
w_freqs: torch.Size([1024, 16])  (width)
Total dim pairs: 48  (== head_dim//2 = 48)


## 4. 3D Frequency Grid Assembly

In `WanModel.forward` (line 377), the three frequency tensors are expanded into a 3D grid (F x H x W) and concatenated along the frequency dimension. Each position `(f, h, w)` in the video grid gets a unique rotation encoding that combines its temporal, height, and width coordinates.

The result has shape `[seq_len, 1, head_dim//2]` where `seq_len = F * H * W`. The `1` in the middle broadcasts over `num_heads` (all heads share the same positional encoding).

Source: `diffsynth/models/wan_video_dit.py`, line 377

In [5]:
# Source: diffsynth/models/wan_video_dit.py, line 377 (used in WanModel.forward)
F, H, W = 4, 4, 4  # small grid for demo
freqs = torch.cat([
    f_freqs[:F].view(F, 1, 1, -1).expand(F, H, W, -1),
    h_freqs[:H].view(1, H, 1, -1).expand(F, H, W, -1),
    w_freqs[:W].view(1, 1, W, -1).expand(F, H, W, -1),
], dim=-1).reshape(F * H * W, 1, -1)
assert freqs.shape == torch.Size([F * H * W, 1, head_dim // 2])
print(f"Grid: {F}x{H}x{W} = {F*H*W} positions")
print(f"Freqs shape: {freqs.shape}  [seq_len, 1, head_dim//2]")
print(f"dtype: {freqs.dtype}")

Grid: 4x4x4 = 64 positions
Freqs shape: torch.Size([64, 1, 48])  [seq_len, 1, head_dim//2]
dtype: torch.complex128


## 5. rope_apply: Applying RoPE to Q/K

The `rope_apply` function rotates each query/key vector by the precomputed angles. The key implementation detail is the **dtype chain** (Pitfall 6):

1. Input `x` arrives as `float32` (or `bfloat16` during training)
2. `x.to(torch.float64)` upcasts before `view_as_complex` -- complex multiplication requires matching precision with the complex128 frequency tensors
3. `torch.view_as_complex` interprets pairs of consecutive floats `(a, b)` as the complex number `a + bi`
4. Complex multiply `x_out * freqs` performs the rotation in the complex plane
5. `torch.view_as_real` converts back to real-valued pairs
6. `.to(x.dtype)` downcasts to the original dtype (float32)

The float64 upcast is required because `torch.view_as_complex` needs matching precision with the complex128 frequency tensors produced by `torch.polar`.

Source: `diffsynth/models/wan_video_dit.py`, line 92

In [6]:
# Source: diffsynth/models/wan_video_dit.py, line 92
B, num_heads, dim = 1, 4, 384
S = F * H * W  # 64 positions from the grid above
q = torch.randn(B, S, dim)  # flat Q from SelfAttention
q_rotated = rope_apply(q, freqs, num_heads)
assert q_rotated.shape == torch.Size([B, S, dim])
assert q_rotated.dtype == q.dtype  # back to original dtype
print(f"Q input:   {q.shape}, dtype={q.dtype}")
print(f"Q rotated: {q_rotated.shape}, dtype={q_rotated.dtype}")
print(f"Values changed: {not torch.allclose(q, q_rotated)}")

Q input:   torch.Size([1, 64, 384]), dtype=torch.float32
Q rotated: torch.Size([1, 64, 384]), dtype=torch.float32
Values changed: True


## 6. Looking Ahead: S2V Per-Token RoPE

The base DiT and S2V (sketch-to-video) model both use 3D RoPE, but with a fundamental difference in how positions are assigned:

- **Base DiT:** positions are **sequential integers** (0, 1, 2, ...) -- every token in the 3D grid gets a position from its `(f, h, w)` coordinates
- **S2V:** positions are **sampled via `np.linspace`** -- each FramePack bucket can have different start/end offsets, enabling variable-rate temporal positioning

We will explore S2V's RoPE in depth in NB-14 (Track 5: S2V Pipeline). For now, the comparison below shows the structural difference.

In [7]:
# Load wan_video_dit_s2v.py for S2V RoPE comparison
# Source: diffsynth/models/wan_video_dit_s2v.py, line 26
_s2v_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit_s2v.py"
_spec_s2v = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit_s2v", _s2v_path)
s2v = importlib.util.module_from_spec(_spec_s2v)
sys.modules["diffsynth.models.wan_video_dit_s2v"] = s2v
_spec_s2v.loader.exec_module(s2v)

import numpy as np

# S2V concatenates freqs into one tensor (not a tuple like base DiT)
freqs_s2v = torch.cat(precompute_freqs_cis_3d(head_dim), dim=1)
print(f"S2V freqs (concatenated): {freqs_s2v.shape}  (all axes in one tensor)")
print(f"Base DiT freqs (tuple):    3 x {f_freqs.shape}  (separate per-axis)")

# S2V uses np.linspace for position sampling (vs integer indexing)
# grid_sizes: [start_offsets, end_positions, total_positions]
grid_sizes = [[
    torch.tensor([0, 0, 0]).unsqueeze(0),
    torch.tensor([F, H, W]).unsqueeze(0),
    torch.tensor([F, H, W]).unsqueeze(0),
]]
x_dummy = torch.randn(1, F*H*W, num_heads, head_dim)
result = s2v.rope_precompute(x_dummy, grid_sizes, freqs_s2v)
# rope_precompute returns complex-valued output: [B, S, N, head_dim//2]
assert result.shape == torch.Size([1, F*H*W, num_heads, head_dim // 2])
print(f"\nBase DiT rope_apply output: {q_rotated.shape}  [B, S, dim] (flat, real-valued)")
print(f"S2V rope_precompute output: {result.shape}  [B, S, N, head_dim//2] (per-head, complex)")
print(f"S2V output dtype: {result.dtype}  (complex -- rotations applied internally)")

S2V freqs (concatenated): torch.Size([1024, 48])  (all axes in one tensor)
Base DiT freqs (tuple):    3 x torch.Size([1024, 16])  (separate per-axis)

Base DiT rope_apply output: torch.Size([1, 64, 384])  [B, S, dim] (flat, real-valued)
S2V rope_precompute output: torch.Size([1, 64, 4, 48])  [B, S, N, head_dim//2] (per-head, complex)
S2V output dtype: torch.complex128  (complex -- rotations applied internally)


The S2V model uses `np.linspace` to sample positions within each FramePack bucket, enabling variable-rate temporal encoding. We will explore this in depth in NB-14 (Track 5: S2V Pipeline). For now, the key takeaway is that base DiT treats positions as fixed grid coordinates, while S2V treats them as learnable sampling points.

## Summary

### Key Takeaways
- **1D RoPE base:** `precompute_freqs_cis` creates a lookup table of complex rotation angles via a geometric frequency series and `torch.polar`. Each position maps to `dim//2` unit complex numbers.
- **Three-axis band split:** `head_dim` is split into (f, h, w) bands. With reduced config (`head_dim=96`), the split is equal `(32, 32, 32)`. In production (`head_dim=128`), it is unequal `(44, 42, 42)` -- the temporal axis absorbs the integer division remainder.
- **Grid assembly:** Three 1D frequency tables are broadcast-expanded into a 3D grid `[F*H*W, 1, head_dim//2]`, giving each video token a unique position encoding.
- **dtype chain in `rope_apply`:** float32 input -> float64 upcast -> view_as_complex -> complex128 multiply -> view_as_real -> float64 -> float32 downcast.

**S2V divisibility validation (XC-06):** The reduced config (`head_dim=96`) satisfies the S2V divisibility constraint: `head_dim//2 = 48, 48 % 3 = 0`.

### Source References

| Symbol | Location |
|--------|----------|
| `precompute_freqs_cis` | `diffsynth/models/wan_video_dit.py`, line 83 |
| `precompute_freqs_cis_3d` | `diffsynth/models/wan_video_dit.py`, line 75 |
| `rope_apply` | `diffsynth/models/wan_video_dit.py`, line 92 |
| `rope_precompute` (S2V) | `diffsynth/models/wan_video_dit_s2v.py`, line 26 |

## Exercises

### Exercise 1 -- Different grid sizes
Change `F, H, W` to `(8, 4, 4)` in the grid assembly cell and reassemble the frequency grid. How does the `seq_len` change? What happens to the temporal dimension's contribution to the concatenated frequencies?

### Exercise 2 -- Theta experiment
Change `theta` from `10000.0` to `1000.0` in `precompute_freqs_cis`. How do the frequency magnitudes change? What does this mean for position sensitivity? (Hint: lower theta means faster oscillation -- positions go through more rotations over a given range.)

### Exercise 3 -- Production band split
Compute the band split for production configs: `head_dim=128` (base model) and `head_dim=64` (smaller model). Which axis absorbs the remainder in each case? Verify with `f_dim + h_dim + w_dim == head_dim`.